&nbsp;

## ChIP-Atlas Diff Analysis Parse Notebook (histone modification)

- 既に ChIP-Atlas `diff Analysis` を実行済みの heat-stress related ChIP-seq dataset に対し、`bedtools intersect` を行い、その結果をparse するための notebook

In [1]:
import os
import datetime
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import polars as pl
from IPython.display import display


os.environ["POLARS_MAX_THREADS"] = "16"

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### この notebook を実行するために必要な手順

1. [ChIP-Atlas Diff Analysis](https://chip-atlas.org/diff_analysis) を実行する
    1. 事前に定義した heat-stress (dataset A) vs. ctrl (dataset B) のデータに従ってペアを作成し、実行する
2. `bedtools intersect -wo`を実行する CWLファイル [../../Workflow/bedtools_intersect_v2_scatter.cwl](../../Workflow/bedtools_intersect_v2_scatter.cwl)を実行する

In [2]:
INTERSECT_DIFF_COLUMNS = [
    # A: ChIP-Atlas diff DPR (8 cols)  <- query (-a)
    "dpr_chrom", "dpr_start", "dpr_end",
    "norm_count_A", "norm_count_B",  # dataset A → Heat stress, B → Control
    "log2fc", "p_value", "q_value",
    # B: fanta.bio CRE (11 cols)       <- database (-b)
    "cre_chrom", "cre_start", "cre_end", "cre_id",
    "cre_score", "cre_strand", "thickStart", "thickEnd",
    "itemRgb", "cre_name", "pooled_cre_scores",
    # -wo
    "overlap_bp",
]

In [12]:
#繰り返し処理を行う
BED_INTERSECT_DIR = pathlib.Path("../../out/ChIP-Atlas_diff/ChIPseq_histone/bedfile_intersect_v2/")

def read_intersect_bed(bed_path: pathlib.Path, q_threshold: float = 0.01) -> pl.DataFrame:
    """
    Read one bedtools intersect -wo output and tag with diff dataset ID from filename.
    default q-value threshold is 0.01.
    """
    # v2 filename: <dataset>_intersecting_<cre-peaks>.bed
    diff_dataset = bed_path.stem.split("_intersecting_", 1)[0]

    return (
        pl.read_csv(
            bed_path,
            separator="\t",
            has_header=False,
            new_columns=INTERSECT_DIFF_COLUMNS,
            # Force float dtypes: leading "1" q-values can make Polars infer Int64
            # and drop scientific-notation significant hits under ignore_errors.
            schema_overrides={
                "dpr_start": pl.Int64,
                "dpr_end": pl.Int64,
                "log2fc": pl.Float64,
                "p_value": pl.Float64,
                "q_value": pl.Float64,
                "cre_start": pl.Int64,
                "cre_end": pl.Int64,
                "cre_score": pl.Int64,
                "thickStart": pl.Int64,
                "thickEnd": pl.Int64,
                "overlap_bp": pl.Int64,
            },
            n_threads=16,
            ignore_errors=True,
        )
        .filter(pl.col("q_value") < q_threshold)
        .with_columns(
            # Fraction of CRE length covered by the DPR overlap (-wo)
            (pl.col("overlap_bp") / (pl.col("cre_end") - pl.col("cre_start"))).alias("overlap_frac_of_fantabio_cre"),
            pl.struct(
                "dpr_start",
                "dpr_end",
                "dpr_chrom"
            ).alias("dpr_interval")
        )
        .with_columns(
            pl.col("pooled_cre_scores").str.extract(r"directionality:([^|]+)", 1).cast(pl.Float64, strict=False).alias("directionality"),
            pl.col("pooled_cre_scores").str.extract(r"class:([^|]+)", 1).alias("fantabio_cre_class"),
        )
        .with_columns(
            pl.lit(diff_dataset).alias("diff_dataset"),
            pl.lit(bed_path.name).alias("source_bed_file"),
        )
        .drop("cre_score", "cre_strand", "thickStart", "thickEnd", "itemRgb", "pooled_cre_scores", "norm_count_A", "norm_count_B", "dpr_start", "dpr_end", "dpr_chrom")
    )

bed_paths = sorted(
    BED_INTERSECT_DIR.glob("*_intersecting_*.bed")
)

all_diff_df = (
    pl.concat(
        [read_intersect_bed(p) for p in bed_paths],
        how="vertical_relaxed",
    )
    .sort(
        "cre_chrom",
        descending=False,
    )
)

# mismatch = all_diff_df.filter(pl.col("cre_chrom") != pl.col("dpr_chrom"))
# print(f"mismatch rows: {mismatch.height}")
pl.Config.set_tbl_rows(10)
pl.Config.set_fmt_str_lengths(100)
print(all_diff_df.select("diff_dataset").n_unique())
print(all_diff_df.select("diff_dataset").unique())
print(f"Diff Analysis detected all unique CREs (q-value < 0.01): {all_diff_df.select("cre_id").n_unique()}")
display(all_diff_df)

2
shape: (2, 1)
┌─────────────────────────────────────┐
│ diff_dataset                        │
│ ---                                 │
│ str                                 │
╞═════════════════════════════════════╡
│ GSE209667_chip_h3k4me3_hs_42C_60min │
│ GSE105028_chip_h3k27ac_hs_42C_60min │
└─────────────────────────────────────┘
Diff Analysis detected all unique CREs (q-value < 0.01): 5648


log2fc,p_value,q_value,cre_chrom,cre_start,cre_end,cre_id,cre_name,overlap_bp,overlap_frac_of_fantabio_cre,dpr_interval,directionality,fantabio_cre_class,diff_dataset,source_bed_file
f64,f64,f64,str,i64,i64,str,str,i64,f64,struct[3],f64,str,str,str
-1.272021,0.00138,0.009665,"""chr1""",1470933,1471201,"""FCHS_550""","""FCHS_550""",68,0.253731,"{1471133,1471409,""chr1""}",0.8,"""ELA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
-1.272021,0.00138,0.009665,"""chr1""",1471165,1471210,"""FCHS_551""","""FCHS_551""",45,1.0,"{1471133,1471409,""chr1""}",1.0,"""ELA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
-1.272021,0.00138,0.009665,"""chr1""",1471244,1471454,"""FCHS_552""","""cp5@ATAD3B""",165,0.785714,"{1471133,1471409,""chr1""}",0.818182,"""ELA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
-1.272021,0.00138,0.009665,"""chr1""",1471361,1471972,"""FCHS_553""","""cp4@ATAD3B""",48,0.07856,"{1471133,1471409,""chr1""}",0.928974,"""PLA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
-1.152493,0.001205,0.008716,"""chr1""",1511351,1511966,"""FCHS_568""","""FCHS_568""",362,0.588618,"{1511344,1511713,""chr1""}",0.28169,"""ELA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
-1.516931,0.000511,0.004434,"""chrX""",154479128,154479549,"""FCHS_446674""","""cp2@LAGE3""",72,0.171021,"{154479477,154479790,""chrX""}",-0.982428,"""PLA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.213783,0.00095,0.007217,"""chrX""",155025776,155026125,"""FCHS_446811""","""cp1@FUNDC2""",94,0.269341,"{155026031,155026506,""chrX""}",1.0,"""PLA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.213783,0.00095,0.007217,"""chrX""",155026411,155026727,"""FCHS_446812""","""cp10@FUNDC2""",95,0.300633,"{155026031,155026506,""chrX""}",0.668966,"""ELA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""


&nbsp;

&nbsp;

&nbsp;

# Selection of Differencially Peaked Region (DPR)

&nbsp;

## (1) Filtering Condition

- `log2(Fold Change) > 0.5`

In [4]:
all_diff_up = all_diff_df.filter(
    pl.col("log2fc") > 0.5
)

display(all_diff_up)

log2fc,p_value,q_value,cre_chrom,cre_start,cre_end,cre_id,cre_name,overlap_bp,overlap_frac_of_fantabio_cre,dpr_interval,directionality,fantabio_cre_class,diff_dataset,source_bed_file
f64,f64,f64,str,i64,i64,str,str,i64,f64,struct[3],f64,str,str,str
1.162215,0.000026,0.000425,"""chr1""",1858682,1858895,"""FCHS_754""","""cp7@GNB1""",213,1.0,"{1858489,1859228,""chr1""}",-0.925156,"""PLA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.162215,0.000026,0.000425,"""chr1""",1859132,1859890,"""FCHS_755""","""cp19@GNB1""",96,0.126649,"{1858489,1859228,""chr1""}",-0.176471,"""ELA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.188244,0.000055,0.000751,"""chr1""",1888971,1889672,"""FCHS_766""","""cp9@GNB1""",291,0.415121,"{1889381,1889797,""chr1""}",-0.823529,"""ELA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.188244,0.000055,0.000751,"""chr1""",1889569,1890175,"""FCHS_767""","""cp1@GNB1""",228,0.376238,"{1889381,1889797,""chr1""}",0.903509,"""ELA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.363975,0.000306,0.002942,"""chr1""",3900053,3900634,"""FCHS_1441""","""cp2@C1orf174,cp2@LINC01134""",200,0.344234,"{3900434,3900725,""chr1""}",-0.764322,"""PLA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1.057816,0.000045,0.000644,"""chrX""",115650343,115650802,"""FCHS_443355""","""FCHS_443355""",69,0.150327,"{115650051,115650412,""chrX""}",0.384615,"""ELA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.823036,0.000006,0.000134,"""chrX""",134604038,134604743,"""FCHS_444890""","""FCHS_444890""",4,0.005674,"{134603634,134604042,""chrX""}",-0.372549,"""ELA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.332147,0.001114,0.008196,"""chrX""",150568174,150568813,"""FCHS_445813""","""cp1@MTM1""",14,0.021909,"{150567879,150568188,""chrX""}",0.997353,"""PLA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""


&nbsp;

&nbsp;

&nbsp;

## Filter by Overlap Fraction

- 対象は直前の `log2fc > 0.5` 通過後（`all_diff_up`）
- 同一 `diff_dataset` × `cre_id` で複数 DPR が重なる場合、`overlap_frac_of_fantabio_cre` が最大の行のみ残す
- check: `FCHS_5` の before / after を表示


In [5]:
# Among upregulated DPRs (log2fc > 0.5), keep max CRE-cover fraction
# per (diff_dataset, cre_id). Ties keep multiple rows.
CHECK_CRE_ID = "FCHS_571903"

fchs5_before = all_diff_up.filter(pl.col("cre_id") == CHECK_CRE_ID).sort(
    "diff_dataset",
    "overlap_frac_of_fantabio_cre",
    descending=[False, True],
)
print(f"[before] {CHECK_CRE_ID}: {fchs5_before.height} rows")
display(fchs5_before)

all_diff_up_filtered = all_diff_up.filter(
    pl.col("overlap_frac_of_fantabio_cre")
    == pl.col("overlap_frac_of_fantabio_cre").max().over("diff_dataset", "cre_id")
)

fchs5_after = all_diff_up_filtered.filter(pl.col("cre_id") == CHECK_CRE_ID).sort(
    "diff_dataset",
    "overlap_frac_of_fantabio_cre",
    descending=[False, True],
)
print(
    f"[after]  {CHECK_CRE_ID}: {fchs5_after.height} rows "
    f"(all_diff_up {all_diff_up.height} -> {all_diff_up_filtered.height})"
)
display(fchs5_after)

[before] FCHS_571903: 0 rows


log2fc,p_value,q_value,cre_chrom,cre_start,cre_end,cre_id,cre_name,overlap_bp,overlap_frac_of_fantabio_cre,dpr_interval,directionality,fantabio_cre_class,diff_dataset,source_bed_file
f64,f64,f64,str,i64,i64,str,str,i64,f64,struct[3],f64,str,str,str


[after]  FCHS_571903: 0 rows (all_diff_up 2434 -> 2229)


log2fc,p_value,q_value,cre_chrom,cre_start,cre_end,cre_id,cre_name,overlap_bp,overlap_frac_of_fantabio_cre,dpr_interval,directionality,fantabio_cre_class,diff_dataset,source_bed_file
f64,f64,f64,str,i64,i64,str,str,i64,f64,struct[3],f64,str,str,str


&nbsp;

&nbsp;

&nbsp;

## After Filtering

In [6]:
print(f"Diff Analysis detected all unique CREs: {all_diff_up_filtered.select("cre_id").n_unique()}")
display(all_diff_up_filtered)

Diff Analysis detected all unique CREs: 2223


log2fc,p_value,q_value,cre_chrom,cre_start,cre_end,cre_id,cre_name,overlap_bp,overlap_frac_of_fantabio_cre,dpr_interval,directionality,fantabio_cre_class,diff_dataset,source_bed_file
f64,f64,f64,str,i64,i64,str,str,i64,f64,struct[3],f64,str,str,str
1.162215,0.000026,0.000425,"""chr1""",1858682,1858895,"""FCHS_754""","""cp7@GNB1""",213,1.0,"{1858489,1859228,""chr1""}",-0.925156,"""PLA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.162215,0.000026,0.000425,"""chr1""",1859132,1859890,"""FCHS_755""","""cp19@GNB1""",96,0.126649,"{1858489,1859228,""chr1""}",-0.176471,"""ELA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.188244,0.000055,0.000751,"""chr1""",1888971,1889672,"""FCHS_766""","""cp9@GNB1""",291,0.415121,"{1889381,1889797,""chr1""}",-0.823529,"""ELA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.188244,0.000055,0.000751,"""chr1""",1889569,1890175,"""FCHS_767""","""cp1@GNB1""",228,0.376238,"{1889381,1889797,""chr1""}",0.903509,"""ELA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.363975,0.000306,0.002942,"""chr1""",3900053,3900634,"""FCHS_1441""","""cp2@C1orf174,cp2@LINC01134""",200,0.344234,"{3900434,3900725,""chr1""}",-0.764322,"""PLA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1.057816,0.000045,0.000644,"""chrX""",115650343,115650802,"""FCHS_443355""","""FCHS_443355""",69,0.150327,"{115650051,115650412,""chrX""}",0.384615,"""ELA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.823036,0.000006,0.000134,"""chrX""",134604038,134604743,"""FCHS_444890""","""FCHS_444890""",4,0.005674,"{134603634,134604042,""chrX""}",-0.372549,"""ELA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.332147,0.001114,0.008196,"""chrX""",150568174,150568813,"""FCHS_445813""","""cp1@MTM1""",14,0.021909,"{150567879,150568188,""chrX""}",0.997353,"""PLA""","""GSE105028_chip_h3k27ac_hs_42C_60min""","""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""


In [7]:
# For figshare
all_diff_up_filtered.unnest(
    # Expand the DPR struct into dpr_start / dpr_end / dpr_chrom columns
    "dpr_interval"
).sort(
    "cre_chrom", "cre_start", "cre_end", "diff_dataset"
).write_csv(
    "../../out/figshare/chip_atlas_diff_analysis/overlapped_fantabio_cre/hs_related_chipseq_histone_diff_up_fantabio_cre.tsv",
    separator="\t",
)

In [8]:
# Aggregate upregulated + max-overlap DPRs to CRE level
all_diff_up_select = all_diff_up_filtered.group_by(
    [
        "cre_chrom",
        "cre_start",
        "cre_end",
        "cre_id",
        "cre_name",
        # "dpr_chrom",
        "directionality",
        "fantabio_cre_class"
    ]
).agg(
    pl.col("overlap_frac_of_fantabio_cre").alias("overlap_frac_of_fantabio_cre"),
    pl.col("dpr_interval").alias("dpr_interval"),
    pl.col("log2fc").alias("log2fc"),
    pl.col("p_value").alias("p_value"),
    pl.col("q_value").alias("q_value"),
    pl.col("diff_dataset").unique().alias("diff_dataset"),
    pl.col("diff_dataset").unique().count().alias("diff_dataset_count"),
    pl.col("source_bed_file").alias("source_bed_file"),
).sort(
    "cre_chrom",
    "cre_start",
    "cre_end",
    descending=False
)

pl.Config.set_tbl_rows(6)
display(all_diff_up_select)

cre_chrom,cre_start,cre_end,cre_id,cre_name,directionality,fantabio_cre_class,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,str,str,f64,str,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""chr1""",1858682,1858895,"""FCHS_754""","""cp7@GNB1""",-0.925156,"""PLA""",[1.0],"[{1858489,1859228,""chr1""}]",[1.162215],[0.000026],[0.000425],"[""GSE105028_chip_h3k27ac_hs_42C_60min""]",1,"[""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""chr1""",1859132,1859890,"""FCHS_755""","""cp19@GNB1""",-0.176471,"""ELA""",[0.126649],"[{1858489,1859228,""chr1""}]",[1.162215],[0.000026],[0.000425],"[""GSE105028_chip_h3k27ac_hs_42C_60min""]",1,"[""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""chr1""",1888971,1889672,"""FCHS_766""","""cp9@GNB1""",-0.823529,"""ELA""",[0.415121],"[{1889381,1889797,""chr1""}]",[1.188244],[0.000055],[0.000751],"[""GSE105028_chip_h3k27ac_hs_42C_60min""]",1,"[""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chrX""",150568174,150568813,"""FCHS_445813""","""cp1@MTM1""",0.997353,"""PLA""",[0.021909],"[{150567879,150568188,""chrX""}]",[1.332147],[0.001114],[0.008196],"[""GSE105028_chip_h3k27ac_hs_42C_60min""]",1,"[""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""chrX""",155025776,155026125,"""FCHS_446811""","""cp1@FUNDC2""",1.0,"""PLA""",[0.269341],"[{155026031,155026506,""chrX""}]",[1.213783],[0.00095],[0.007217],"[""GSE105028_chip_h3k27ac_hs_42C_60min""]",1,"[""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""chrX""",155026411,155026727,"""FCHS_446812""","""cp10@FUNDC2""",0.668966,"""ELA""",[0.300633],"[{155026031,155026506,""chrX""}]",[1.213783],[0.00095],[0.007217],"[""GSE105028_chip_h3k27ac_hs_42C_60min""]",1,"[""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"


In [9]:
all_diff_up_select.write_parquet(
    "../../out/ChIP-Atlas_diff/diff_analysis_filter_parquet/only_cre/fantabio_cre_list_diff_analysis_chipseq_histone.parquet",
    use_pyarrow=True
)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Intergration (upregulated transcripts)

In [10]:
# up5p transcript lists + CRE
fantabio_HN2_up5p_transcripts_cre = pl.read_parquet(
    "../../out/fantabio/parquet_file/up5p_transcript_fantabio_v261001.parquet"
).join(
    all_diff_up_select,
    on=["cre_id", "cre_chrom", "cre_name", "directionality", "fantabio_cre_class"],
    how="inner",
    coalesce=True
)

fantabio_HN2_up5p_transcripts_cre.write_parquet(
    "../../out/ChIP-Atlas_diff/diff_analysis_filter_parquet/transcript_cre/fantabio_HN2_up5p_v261107_diff_analysis_chipseq_histone.parquet",
    use_pyarrow=True
)

print(f"up5p_fantabio transcript count: {fantabio_HN2_up5p_transcripts_cre.select('transcript_id').n_unique()}")
print(f"up5p_fantabio lncRNA transcript count: {fantabio_HN2_up5p_transcripts_cre.filter(pl.col("transcript_type") == "lncRNA").select('transcript_id').n_unique()}")
print(f"up5p_fantabio proximal CRE + Diff Analysis count: {fantabio_HN2_up5p_transcripts_cre.select('cre_id').n_unique()}")
print(f"up5p_fantabio transcript-CRE pairs count: {fantabio_HN2_up5p_transcripts_cre.select(['transcript_id', 'cre_id']).n_unique()}")
pl.Config.set_tbl_rows(4)
display(fantabio_HN2_up5p_transcripts_cre)

up5p_fantabio transcript count: 943
up5p_fantabio lncRNA transcript count: 134
up5p_fantabio proximal CRE + Diff Analysis count: 476
up5p_fantabio transcript-CRE pairs count: 1525


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,cre_start,cre_end,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,i64,i64,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""ENST00000540260.1""",328,9,69,319,216,1,34,215,"""FKBP4""","""HGNC:3720""","""protein_coding_CDS_not_defined""","""ENSG00000004478.9""","[""non_canonical_TEC""]","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""non canonical TEC""]",1,"""FCHS_86240""","""chr12""",2794544,"""2795205""","""cp3@FKBP4,cp1@ITFG2-AS1""","[""ENST00000839341.1"", ""ENST00000839340.1"", … ""DQ893668""]","""0""",452,"[{""EH38E1587359"",""pELS""}, {""EH38E2998737"",""PLS""}]","[{""HSF1"",3307,[{""SRX16646111"",3307}, {""SRX16646095"",3215}, … {""SRX3322007"",507}]}, {""ONECUT2"",2305,[{""SRX3927334"",2305}, {""SRX3927332"",713}, … {""SRX2731587"",570}]}, … {""FLI1"",508,[{""SRX117011"",508}]}]",null,"[{""hg19::chr12:2903742..2903763,-;hg_128965.1"",""hg_128965.1""}, {""hg19::chr12:2903775..2903794,-;hg_128966.1"",""hg_128966.1""}, … {""hg19::chr12:2904305..2904316,+;hg_123449.1"",""p6@FKBP4""}]",null,0.996397,"""PLA""","""204,0,0""","""red""",2794543,2795205,[0.217523],"[{2793910,2794687,""chr12""}]",[1.170049],[0.000014],[0.000248],"[""GSE105028_chip_h3k27ac_hs_42C_60min""]",1,"[""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""ENST00000676231.2""",324,7,75,317,209,2,40,207,"""HSPB1""","""HGNC:5246""","""protein_coding""","""ENSG00000106211.11""","[""inferred_exon_combination"", ""RNA_Seq_supported_only"", ""basic""]","""+""",3,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""intropolis_filtered"", ""Snaptron_GTEX"", … ""RNA-Seq supported only""]",4,"""FCHS_383680""","""chr7""",76302016,"""76302438""","""FCHS_383680""","[""MP202714""]","""120""",235,"[{""EH38E3781182"",""pELS""}, {""EH38E3781183"",""PLS""}]","[{""HSF1"",3261,[{""SRX16646111"",3261}]}, {""ESR1"",1776,[{""SRX21439722"",1776}, {""SRX21439724"",1556}, … {""SRX21439743"",559}]}, … {""GATA3"",518,[{""SRX100559"",518}]}]","[{""HS_00093581"",""Fetal Intestine Small"",""Small intestine"",""Intestine"",""SVM""}]",null,null,0.584127,"""ELA""","""255,202,74""","""yellow""",76302015,76302438,[0.992908],"[{76302006,76302435,""chr7""}]",[1.647013],[0.000008],[0.000164],"[""GSE105028_chip_h3k27ac_hs_42C_60min""]",1,"[""GSE105028_chip_h3k27ac_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000906067.1""",86,42,278,44,54,32,165,22,"""YARS1""","""HGNC:12840""","""protein_coding""","""ENSG00000134684.14""","[""basic"", ""TAGENE""]","""-""",2,null,null,"[""polya_site"", ""polya_signal""]","""ensembl_havana_gene_homo_sapiens""","[""DanaFarber""]",1,"""FCHS_9889""","""chr1""",32816778,"""32817049""","""cp4@S100PBP""","[""ENST00000530710.5""]","""-11""",-315,"[{""EH38E1335378"",""pELS""}]","[{""ATF4"",3560,[{""SRX10900660"",3560}, {""SRX10900663"",3324}, {""SRX10900664"",3030}]}, {""MYC"",2607,[{""SRX6935381"",2607}]}, … {""GATA1"",609,[{""SRX4172745"",609}]}]","[{""HS_00004119"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce

In [11]:
# for figshare
# シンプルな TSV形式で保存する
fantabio_HN2_up5p_transcripts_cre.select(
    "transcript_id",
    "gene_id",
    "gene_symbol",
    "HN2",
    "HN2_stranded",
    "transcript_type",
    "cre_id",
    "cre_name",
    "transcript_distance",
    "diff_dataset_count",
    "diff_dataset"
).with_columns(
    # Convert list[str] to a single string so it can be written to TSV
    # Deduplicate and sort first so the joined string is stable across runs
    pl.col("diff_dataset").list.unique().list.sort().list.join("; ")
).unique().sort(
    by=["HN2", "diff_dataset_count"],
    descending=True
).write_csv(
    "../../out/figshare/chip_atlas_diff_analysis/overlapped_fantabio_cre_transcript_pair/hs_related_chipseq_histone_diff_up_fantabio_cre_up5p_transcript_pair.tsv",
    separator="\t"
)